# SM-SIP: Zero-Shot Inference & Evaluation (8-bit Quantized)

## Semantic & Multilingual Salient Information Prompting

This notebook implements the **zero-shot inference pipeline** for Italian text summarization using **8-bit quantization**:

1. **Load** the fine-tuned SigExt model from Hugging Face
2. **Generate** a held-out test set (never seen during training)
3. **Run** the full pipeline: SigExt → Llama-3 → Evaluation Metrics
4. **Export** results to JSON for analysis

### Key Difference from 4-bit Version
This notebook uses **8-bit quantization** instead of 4-bit:
- **Higher precision**: 8-bit maintains better model quality
- **More memory usage**: ~8GB vs ~4GB for the LLM
- **Faster inference**: Less computational overhead from quantization

### Configuration
- **SigExt Model**: `LookUpMark/sigext-wits-it-10k-060t`
- **LLM**: `meta-llama/Llama-3.1-8B-Instruct` (8-bit quantized)
- **Test Samples**: 500 articles (skipping training data)

---
## 1. Environment Setup

Install all required dependencies and authenticate with Hugging Face.

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10'

# Download Italian Spacy model
!python -m spacy download it_core_news_sm

In [ ]:
import os
import torch
import json
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

---
## 2. Configuration

Define model paths and authentication.

Token is loaded from:
1. **Kaggle**: Using `kaggle_secrets` (for Kaggle notebooks)
2. **Environment**: Using `HF_TOKEN` environment variable (for local/Colab)

In [ ]:
# Hugging Face authentication
# Option 1: Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    # Option 2: Environment variable or manual input
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

# Configuration
CONFIG = {
    "SIGEXT_MODEL_ID": "LookUpMark/sigext-wits-it-10k-060t",
    "LLAMA_MODEL_ID": "meta-llama/Llama-3.1-8B-Instruct",
    "NUM_TEST_SAMPLES": 500,
    "MAX_LEN": 2048,
    "SKIP_TRAIN_SAMPLES": 10000  # Skip first 10k samples used for training
}

# Authenticate with Hugging Face
login(token=HF_TOKEN)

---
## 3. Generate Held-Out Test Set

Load the WITS dataset and create a test set that was **never seen during training**.
We skip the first 10,000 samples to avoid data contamination.

In [ ]:
def get_test_data():
    """Generate a held-out test set from WITS dataset."""
    print(f"--- Generating Test Set ({CONFIG['NUM_TEST_SAMPLES']} new articles) ---")
    print(f"    Skipping first {CONFIG['SKIP_TRAIN_SAMPLES']} samples to avoid contamination.")
    
    # Load dataset in streaming mode
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    
    # Skip training data
    dataset = dataset.skip(CONFIG['SKIP_TRAIN_SAMPLES'])
    
    test_data = []
    print("    Downloading and filtering...")
    pbar = tqdm(total=CONFIG['NUM_TEST_SAMPLES'])
    
    for entry in dataset:
        source = entry['source']
        summary = entry['summary']
        
        # Quality filters
        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue
            
        test_data.append({
            "source": source,
            "reference": summary
        })
        pbar.update(1)
        
        if len(test_data) >= CONFIG['NUM_TEST_SAMPLES']:
            break
            
    pbar.close()
    print(f"    Test Set ready: {len(test_data)} examples.")
    return test_data

test_data = get_test_data()

---
## 4. Load Models

### 4.1 SigExt Model
Load the fine-tuned Salient Information Extractor from Hugging Face.

### 4.2 Llama-3 (8-bit Quantized)
Load Llama-3-8B-Instruct with **8-bit quantization** for efficient inference.

8-bit quantization offers a good balance between memory savings and model quality.

In [ ]:
def load_models():
    """Load SigExt and Llama-3 models."""
    print("--- Loading Models ---")
    
    # A. Load SigExt from Hugging Face
    print(f"1. Downloading SigExt: {CONFIG['SIGEXT_MODEL_ID']}...")
    try:
        sigext_tokenizer = AutoTokenizer.from_pretrained(CONFIG['SIGEXT_MODEL_ID'])
        sigext_model = AutoModelForTokenClassification.from_pretrained(
            CONFIG['SIGEXT_MODEL_ID']
        ).to("cpu")
        print("    -> SigExt loaded successfully!")
    except Exception as e:
        print(f"    ERROR loading SigExt: {e}")
        print("    Verify the repo exists and is accessible with your token.")
        raise e

    # B. Load Llama-3 (8-bit Quantized)
    print(f"2. Loading Llama-3: {CONFIG['LLAMA_MODEL_ID']} (8-bit)...")
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )
    llama_model = AutoModelForCausalLM.from_pretrained(
        CONFIG['LLAMA_MODEL_ID'], 
        quantization_config=bnb_config, 
        device_map="auto"
    )
    llama_tokenizer = AutoTokenizer.from_pretrained(CONFIG['LLAMA_MODEL_ID'])
    llama_tokenizer.pad_token = llama_tokenizer.eos_token
    print("    -> Llama-3 loaded successfully!")
    
    return sigext_model, sigext_tokenizer, llama_model, llama_tokenizer

sigext_model, sigext_tokenizer, llama_model, llama_tokenizer = load_models()

---
## 5. LangChain Pipeline Setup (Zero-Shot)

Configure the LangChain pipeline with a **zero-shot prompt template**.
The model receives no examples, only instructions and the extracted keyphrases.

In [ ]:
def setup_zero_shot_chain(llama_model, llama_tokenizer):
    """Create a zero-shot LangChain pipeline."""
    
    # Create HuggingFace pipeline
    pipe = pipeline(
        "text-generation", 
        model=llama_model, 
        tokenizer=llama_tokenizer, 
        max_new_tokens=256, 
        temperature=0.1
    )
    
    # Wrap in LangChain
    llm = HuggingFacePipeline(pipeline=pipe)
    
    # Zero-shot prompt template (RGC Framework + Structural Priming)
    template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
# RUOLO
Sei un assistente editoriale esperto specializzato nella sintesi di testi complessi per un pubblico accademico e professionale italiano.

# OBIETTIVO
Generare un riassunto astrattivo di alta qualità che catturi l'essenza del testo originale, garantendo fluidità e coerenza logica.

# VINCOLI E CONTESTO
1. **Lingua**: L'output deve essere esclusivamente in Italiano corretto.
2. **Salienza**: Le "Frasi Chiave" fornite sono i punti nodali del discorso. È obbligatorio integrarle nel testo.
3. **Struttura**: Non limitarti a citare le frasi chiave; usale come **scaletta logica** per costruire la narrazione. Il riassunto deve fluire da un punto chiave all'altro in modo naturale.
4. **Stile**: Mantieni un tono formale, distaccato e oggettivo.
<|eot_id|><|start_header_id|>user<|end_header_id|>
Testo Originale:
{source}

Frasi Chiave (da usare come ossatura):
{keyphrases}

Genera ora il riassunto collegando logicamente i punti chiave:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    prompt = PromptTemplate(template=template, input_variables=["source", "keyphrases"])
    
    chain = prompt | llm | StrOutputParser()
    
    print("Zero-shot chain configured successfully!")
    return chain

chain = setup_zero_shot_chain(llama_model, llama_tokenizer)

---
## 6. Keyphrase Extraction Function

Use the SigExt model to extract salient tokens from the source text.
These keyphrases will be injected into the LLM prompt to guide generation.

In [ ]:
def extract_keyphrases(text, model, tokenizer):
    """Extract salient keyphrases using SigExt model."""
    # Tokenize input
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=CONFIG["MAX_LEN"]
    ).to("cpu")
    
    # Run inference
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Get predictions (0 = non-salient, 1 = salient)
    preds = torch.argmax(logits, dim=2)[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Extract tokens marked as salient
    extracted_tokens = [t for t, label in zip(tokens, preds) if label == 1]
    
    # Decode to text
    decoded = tokenizer.decode(tokenizer.convert_tokens_to_ids(extracted_tokens))
    return decoded

---
## 7. Evaluation Metrics

Evaluate the generated summaries using three complementary metrics:

- **ROUGE-1**: Lexical overlap (unigram precision/recall)
- **BERTScore**: Semantic similarity using BERT embeddings
- **KIR (Keyphrase Inclusion Rate)**: Measures prompt obedience

In [ ]:
def evaluate(test_data, sigext_model, sigext_tokenizer, chain):
    """Run full evaluation on test set."""
    print(f"--- Starting Evaluation on {len(test_data)} articles ---")
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    
    metrics = {"bert": [], "rouge": [], "kir": []}
    
    print("    Processing (Llama-3 8-bit is generating)...")
    for item in tqdm(test_data):
        try:
            # 1. SigExt: Extract keyphrases
            keys_text = extract_keyphrases(item['source'], sigext_model, sigext_tokenizer)
            keys_list = [k.strip() for k in keys_text.split() if len(k) > 3]
            
            # 2. Llama-3: Generate summary
            res = chain.invoke({"source": item['source'], "keyphrases": keys_text})
            gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
            
            # 3. Calculate metrics
            # ROUGE-1
            metrics["rouge"].append(
                scorer.score(item['reference'], gen_summary)['rouge1'].fmeasure
            )
            
            # BERTScore
            _, _, F1 = bert_score(
                [gen_summary], [item['reference']], lang="it", verbose=False
            )
            metrics["bert"].append(F1.mean().item())
            
            # KIR (Keyphrase Inclusion Rate)
            if keys_list:
                gen_lower = gen_summary.lower()
                hits = sum(1 for k in keys_list if k.lower() in gen_lower)
                metrics["kir"].append(hits / len(keys_list))
            else:
                metrics["kir"].append(0.0)
                
        except Exception as e:
            print(f"    ! Error on sample: {e}")
            continue

    return metrics

---
## 8. Run Evaluation & Export Results

Run the evaluation and save results to a JSON file for later analysis.

In [ ]:
# Run evaluation
metrics = evaluate(test_data, sigext_model, sigext_tokenizer, chain)

# Compute statistics and build results object
results = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "inference_type": "zero-shot",
        "quantization": "8-bit",
        "sigext_model": CONFIG["SIGEXT_MODEL_ID"],
        "llm_model": CONFIG["LLAMA_MODEL_ID"],
        "num_test_samples": CONFIG["NUM_TEST_SAMPLES"],
        "skip_train_samples": CONFIG["SKIP_TRAIN_SAMPLES"]
    },
    "metrics": {
        "bert_score": {
            "mean": float(np.mean(metrics['bert'])),
            "std": float(np.std(metrics['bert'])),
            "min": float(np.min(metrics['bert'])),
            "max": float(np.max(metrics['bert']))
        },
        "rouge1": {
            "mean": float(np.mean(metrics['rouge'])),
            "std": float(np.std(metrics['rouge'])),
            "min": float(np.min(metrics['rouge'])),
            "max": float(np.max(metrics['rouge']))
        },
        "kir": {
            "mean": float(np.mean(metrics['kir'])),
            "std": float(np.std(metrics['kir'])),
            "min": float(np.min(metrics['kir'])),
            "max": float(np.max(metrics['kir']))
        }
    },
    "raw_scores": {
        "bert": [float(x) for x in metrics['bert']],
        "rouge": [float(x) for x in metrics['rouge']],
        "kir": [float(x) for x in metrics['kir']]
    }
}

# Save to JSON
output_file = f"results_zero_shot_8bit_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

# Display results
print("\n" + "="*55)
print("   ZERO-SHOT INFERENCE RESULTS (8-BIT QUANTIZATION)")
print("="*55)
print(f"   BERTScore: {results['metrics']['bert_score']['mean']:.4f} ± {results['metrics']['bert_score']['std']:.4f}")
print(f"   ROUGE-1:   {results['metrics']['rouge1']['mean']:.4f} ± {results['metrics']['rouge1']['std']:.4f}")
print(f"   KIR:       {results['metrics']['kir']['mean']:.2%} ± {results['metrics']['kir']['std']:.2%}")
print("="*55)
print(f"\n   Results saved to: {output_file}")